# 01. 첫 백테스트 — 보험주 vs 비보험주

**질문:** 보험업 지표/리스크 선별 사고방식이 종목 선택에 도움이 되는가?

| 그룹 | 가설 | 내용 |
|---|---|---|
| 보험주 | H1 | 합산비율 개선 보험사가 시장을 이긴다 → **FISIS 데이터 필요, 02번 노트북에서** |
| 보험주 | H2 | 저PBR·고ROE 보험사 3곳이 보험주 전체를 이긴다 |
| 비보험주 | H1 | 저PER·고ROE 10종목이 KOSPI를 이긴다 |
| 비보험주 | H2 | 변동성 큰 종목·적자 종목을 걸러내면 낙폭이 줄어든다 (언더라이팅식 선별) |

**방식:** 매년 1월 첫 거래일 리밸런싱 · 동일가중 · 1년 보유 · 월 단위로 성과 측정

**한계(리포트에 꼭 적기):** 배당 미포함 · 상장폐지 종목은 마지막 가격으로 고정 · EPS는 해당 시점 KRX 기준값


## 0. 설정 — `.env`에 KRX 아이디/비번이 있어야 데이터가 받아져요

In [ ]:
from dotenv import load_dotenv
load_dotenv()   # .env 의 KRX_ID, KRX_PW 를 먼저 불러온 뒤 pykrx를 import

import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pykrx import stock

START_YEAR, END_YEAR = 2016, 2025   # 리밸런싱 연도
COST = 0.003                        # 리밸런싱 1회당 거래비용(수수료+세금+슬리피지) 0.3%
RF = 0.03                           # 무위험이자율 (샤프 비율용)
TOP_N_UNIVERSE = 100                # 비보험 유니버스: KOSPI 시총 상위 100
os.makedirs("data", exist_ok=True)

## 1. 데이터 받기 (처음 한 번만 오래 걸리고, 이후엔 data/ 폴더 캐시를 씀)

In [ ]:
def biz_day(yyyymmdd):
    """해당 날짜 이후 가장 가까운 거래일"""
    return stock.get_nearest_business_day_in_a_week(yyyymmdd, prev=False)

# 월초 거래일 목록 (변동성 계산 위해 1년 전부터)
months = pd.date_range(f"{START_YEAR-1}-01-01", f"{END_YEAR+1}-01-01", freq="MS")
dates = []
for m in months:
    d = biz_day(m.strftime("%Y%m%d"))
    if d <= pd.Timestamp.today().strftime("%Y%m%d"):
        dates.append(d)
print(len(dates), "개 월초 거래일:", dates[0], "~", dates[-1])

In [ ]:
def load_cached(name, fn):
    path = f"data/{name}.pkl"
    if os.path.exists(path):
        return pd.read_pickle(path)
    df = fn()
    df.to_pickle(path)
    time.sleep(0.5)   # KRX 서버에 무리 주지 않기
    return df

closes, caps = {}, {}
for d in dates:
    df = load_cached(f"cap_{d}", lambda: stock.get_market_cap(d, market="KOSPI"))
    closes[d] = df["종가"]
    caps[d] = df["시가총액"]

close = pd.DataFrame(closes).T.replace(0, np.nan)   # 행=날짜, 열=종목코드
cap = pd.DataFrame(caps).T
close = close.ffill()   # 상장폐지/거래정지 → 마지막 가격 유지 (한계로 명시)
print(close.shape)

In [ ]:
reb_dates = [d for d in dates if d[4:6] == "01" and START_YEAR <= int(d[:4]) <= END_YEAR + 1]
fund = {d: load_cached(f"fund_{d}", lambda: stock.get_market_fundamental(d, market="KOSPI"))
        for d in reb_dates}

kospi = stock.get_index_ohlcv(dates[0], dates[-1], "1001")["종가"]
kospi.index = kospi.index.strftime("%Y%m%d")
kospi = kospi.reindex(dates).ffill()
print("리밸런싱일:", reb_dates)

## 2. 보험주 유니버스 — 이름이 맞는지 꼭 눈으로 확인하기

In [ ]:
INSURERS = ["000810", "005830", "001450", "000370", "000400", "000540",   # 손보
            "032830", "088350", "085620", "082640",                       # 생보
            "003690"]                                                     # 재보험
for t in INSURERS:
    try:
        print(t, stock.get_market_ticker_name(t))
    except Exception as e:
        print(t, "확인 실패 →", e)

## 3. 가설별 종목 선택 규칙

In [ ]:
def roe_table(d):
    f = fund[d].copy()
    f = f[(f["PER"] > 0) & (f["BPS"] > 0)]      # 적자·자본잠식 제외
    f["ROE"] = f["EPS"] / f["BPS"]
    return f

def universe_big(d):
    """비보험 유니버스: 시총 상위 N (보험주 제외)"""
    c = cap.loc[d].drop(INSURERS, errors="ignore").dropna()
    return list(c.sort_values(ascending=False).index[:TOP_N_UNIVERSE])

def pick_nonins_h1(d, n=10):
    """저PER + 고ROE 순위 합산 상위 n"""
    f = roe_table(d).reindex(universe_big(d)).dropna(subset=["PER", "ROE"])
    score = f["PER"].rank() + f["ROE"].rank(ascending=False)
    return list(score.nsmallest(n).index)

def pick_nonins_h2(d):
    """언더라이팅식 선별: 적자 제외 + 최근 12개월 변동성 상위 30% 제외, 나머지 동일가중"""
    uni = universe_big(d)
    ok = set(roe_table(d).index)                       # 흑자·자본 양수
    i = dates.index(d)
    hist = close.iloc[max(0, i-12):i+1][uni].pct_change().dropna(how="all")
    vol = hist.std()
    keep = vol[vol <= vol.quantile(0.7)].index
    return [t for t in uni if t in ok and t in keep]

def pick_ins_all(d):
    return [t for t in INSURERS if t in close.columns and not np.isnan(close.loc[d, t])]

def pick_ins_h2(d, n=3):
    """보험주 중 저PBR + 고ROE 순위 합산 상위 n"""
    f = roe_table(d).reindex(pick_ins_all(d)).dropna(subset=["PBR", "ROE"])
    score = f["PBR"].rank() + f["ROE"].rank(ascending=False)
    return list(score.nsmallest(n).index)

STRATS = {"Ins_All": pick_ins_all, "Ins_H2": pick_ins_h2,
          "NonIns_Uni": universe_big, "NonIns_H1": pick_nonins_h1, "NonIns_H2": pick_nonins_h2}

## 4. 백테스트 엔진 — 연 1회 동일가중 매수 후 1년 보유, 월별 평가

In [ ]:
def backtest(pick_fn):
    values, holdings = [], {}
    v = 1.0
    for k, r in enumerate(reb_dates[:-1]):
        nxt = reb_dates[k+1]
        tickers = pick_fn(r)
        holdings[r[:4]] = tickers
        v *= (1 - COST)
        period = [d for d in dates if r <= d <= nxt]
        rel = close.loc[period, tickers] / close.loc[r, tickers]   # 매수가 대비 배수
        path = v * rel.mean(axis=1)
        values.append(path if not values else path.iloc[1:])
        v = path.iloc[-1]
    return pd.concat(values), holdings

def stats(s):
    r = s.pct_change().dropna()
    yrs = len(r) / 12
    cagr = (s.iloc[-1] / s.iloc[0]) ** (1/yrs) - 1
    vol = r.std() * np.sqrt(12)
    mdd = (s / s.cummax() - 1).min()
    return pd.Series({"CAGR": cagr, "Vol": vol, "Sharpe": (cagr - RF) / vol, "MDD": mdd})

results, picks = {}, {}
for name, fn in STRATS.items():
    results[name], picks[name] = backtest(fn)
results["KOSPI"] = kospi.loc[results["Ins_All"].index] / kospi.loc[reb_dates[0]]

curve = pd.DataFrame(results)
table = curve.apply(stats).T
table.style.format("{:.1%}", subset=["CAGR", "Vol", "MDD"]).format("{:.2f}", subset=["Sharpe"])

## 5. 그래프 & 연도별 보유종목

In [ ]:
curve.index = pd.to_datetime(curve.index)
ax = curve.plot(figsize=(10, 5), logy=True, title="Cumulative value (log scale)")
ax.set_xlabel(""); ax.grid(alpha=.3)
plt.show()

for name in ["Ins_H2", "NonIns_H1"]:
    print(f"\n[{name}] 연도별 보유종목")
    for y, ts in picks[name].items():
        print(y, ", ".join(stock.get_market_ticker_name(t) for t in ts))

## 6. 해석 메모 (experiments.md 에 옮겨 적기)

- Ins_H2 vs Ins_All : 일반 지표(PBR·ROE)로 보험주를 고르는 게 의미 있었나?
- NonIns_H1 vs KOSPI : 가치 전략이 시장을 이겼나?
- NonIns_H2 vs NonIns_Uni : **수익은 비슷한데 MDD가 줄었나?** ← 언더라이팅식 선별의 핵심 지표
- 결과가 기대와 다르면 그 이유 가설 1~2개 적어두기

**다음:** 02번 노트북 — FISIS 합산비율 데이터로 보험주 H1
